In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas
from mpl_toolkits.axes_grid1 import make_axes_locatable

from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.toto.circuitlens import CircuitLensToto
from tsfm_lens.utils import (
    apply_custom_style,
    collect_attributions,
    load_dyst_samples,
)


In [ ]:
apply_custom_style("../../config/plotting.yaml")


In [ ]:
apply_custom_style("../config/plotting.yaml")


In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")


In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../figs", "head_outputs")
os.makedirs(plot_save_dir, exist_ok=True)


In [ ]:
split_name = "gift-eval"

subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Lorenz"
    # ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

    context_length = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length

    # Prepare input series
    sample_idx = 0
    selected_dim = 0
    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :]).unsqueeze(0)
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context = dyst_coords[:, context_start_time:context_end_time]

    print(context.shape)

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
    print(f"future_vals shape: {future_vals.shape}")

elif split_name == "gift-eval":
    # Load the path to gift-eval data, which is stored in .env file
    split_dir = os.path.join(WORK_DIR, DATA_DIR, split_name)
    # system_name = "electricity/D"
    system_name = "m4_monthly"

    to_univariate = False
    term = "long"

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)
    print("Dataset frequency: ", dataset.freq)
    print("Prediction length: ", dataset.prediction_length)
    print("Number of windows in the rolling evaluation: ", dataset.windows)

    test_split_iter = dataset.test_data
    test_data = next(iter(test_split_iter))

    test_input_split_iter = dataset.test_data.input

    input_entry = next(iter(test_input_split_iter))
    print("Keys in the test data: ", input_entry.keys())

    print("\n\nContext Item id: ", input_entry["item_id"])
    print("Context Start Date: ", input_entry["start"])
    print("Context Frequency: ", input_entry["freq"])
    print(f"Context shape: {input_entry['target'].shape}")

    test_label_split_iter = dataset.test_data.label
    label_entry = next(iter(test_label_split_iter))
    print("\n\nForecast Item id: ", label_entry["item_id"])
    print("Forecast Start Date: ", label_entry["start"])
    print("Forecast Frequency: ", label_entry["freq"])
    print(f"Forecast shape: {label_entry['target'].shape}")

    fig, ax = plt.subplots(figsize=(5, 2))
    context_series = to_pandas(test_data[0])
    future_series = to_pandas(test_data[1])
    prediction_length = future_series.shape[0]
    print(f"prediction length: {prediction_length}")
    context_series.plot(color="black", linewidth=1, ax=ax)
    future_series.plot(color="tab:blue", linewidth=1, ax=ax)
    ax.grid(which="both")
    # ax.legend("ground truth", loc="upper left")
    plt.show()
    num_nans_context = context_series.isna().sum()
    num_nans_future_vals = future_series.isna().sum()

    context = torch.tensor(context_series.values, dtype=torch.float32).unsqueeze(0)
    future_vals = torch.tensor(future_series.values, dtype=torch.float32).unsqueeze(0)
    print(f"num nans context: {num_nans_context}")
    print(f"num nans future vals: {num_nans_future_vals}")
    print(f"context length: {context.shape}")
    print(f"future vals shape: {future_vals.shape}")


In [ ]:
pipeline = CircuitLensToto("Datadog/Toto-Open-Base-1.0", device_map=device)
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")


In [ ]:
num_samples = 10
do_sample = num_samples > 1

pipeline.remove_all_hooks()
pipeline.reset_attribution_inputs_and_outputs()

print("Adding head attribution hooks")
pipeline.add_head_attribution_hooks(
    [(layer_idx, head_idx) for layer_idx in range(num_layers) for head_idx in range(num_heads)]
)

print("Adding MLP attribution hooks")
pipeline.add_mlp_attribution_hooks([layer_idx for layer_idx in range(num_layers)])

print("Adding read stream hooks")
pipeline.add_read_stream_hooks([layer_idx for layer_idx in range(num_layers)])


In [ ]:
pred = pipeline.predict(
    context=context,
    prediction_length=prediction_length,
    num_samples=num_samples,
    samples_per_batch=num_samples,
)


In [ ]:
context_np = context.squeeze().cpu().numpy()
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.squeeze().cpu().numpy()
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)
print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")


In [ ]:
preds = pred.detach().cpu().numpy()
if preds.ndim == 1:
    preds = preds[None, None, :]
elif preds.ndim == 2:
    preds = preds[None, ...]
num_variates = preds.shape[0]
num_generated_samples = preds.shape[1]
prediction_length = preds.shape[2]

fig, axes = plt.subplots(num_variates, 1, figsize=(6, 2 * num_variates))
axes = [axes] if num_variates == 1 else axes

for dim, ax in enumerate(axes):
    ax.plot(context_timesteps, context_np if context_np.ndim == 1 else context_np[dim], color="black", linewidth=1, label="Context")
    ax.plot(future_timesteps, future_vals_np if future_vals_np.ndim == 1 else future_vals_np[dim], color="black", linewidth=1, linestyle="--", label="Ground Truth")
    for sample_idx in range(num_generated_samples):
        ax.plot(
            future_timesteps,
            preds[dim, sample_idx],
            color=DEFAULT_COLORS[sample_idx % len(DEFAULT_COLORS)],
            linewidth=1,
            alpha=0.4,
        )
    median_forecast = np.median(preds[dim], axis=0)
    ax.plot(future_timesteps, median_forecast, color="tab:blue", linewidth=2, label="Median")
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    ax.legend(loc="upper left", frameon=True)

plt.tight_layout()


In [ ]:
head_outputs: dict[int, list[torch.Tensor]] = {}
for layer_idx in range(num_layers):
    layer_outputs = pipeline.head_attribution_outputs.get(layer_idx)
    if layer_outputs is None:
        raise ValueError(f"Missing head attribution outputs for layer {layer_idx}")
    head_outputs[layer_idx] = []
    for head_idx in range(num_heads):
        head_tensors = layer_outputs.get(head_idx)
        if not head_tensors:
            raise ValueError(f"Missing outputs for layer {layer_idx}, head {head_idx}")
        head_outputs[layer_idx].append(collect_attributions(head_tensors).detach().cpu())

mlp_outputs = {layer_idx: collect_attributions(outputs).detach().cpu() for layer_idx, outputs in pipeline.mlp_attribution_outputs.items()}
read_stream_outputs = {layer_idx: collect_attributions(outputs).detach().cpu() for layer_idx, outputs in pipeline.read_stream_outputs.items()}


In [ ]:
def normalize_outputs(outputs: dict[int, list[torch.Tensor]]):
    normalized: dict[int, list[torch.Tensor]] = {}
    for layer_idx, tensors in outputs.items():
        normalized[layer_idx] = []
        for tensor in tensors:
            norms = tensor.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            normalized[layer_idx].append(tensor / norms)
    return normalized

head_outputs_normalized = normalize_outputs(head_outputs)
num_samples, num_timesteps, d_model = next(iter(head_outputs_normalized.values()))[0].shape

sa_gramians = torch.zeros(num_layers, num_samples, num_timesteps, num_heads, num_heads)
for layer in range(num_layers):
    for h1 in range(num_heads):
        for h2 in range(num_heads):
            sa_gramians[layer, :, :, h1, h2] = torch.einsum(
                "b t d, b t d -> b t", head_outputs_normalized[layer][h1], head_outputs_normalized[layer][h2]
            )


In [ ]:
sa_singular_values = torch.zeros(num_layers, num_samples, num_timesteps, num_heads)
for layer in range(num_layers):
    for sample in range(num_samples):
        for timestep in range(num_timesteps):
            sa_singular_values[layer, sample, timestep] = torch.linalg.svd(sa_gramians[layer, sample, timestep]).S

sa_p_vals = torch.sqrt(sa_singular_values)
sa_p_vals = sa_p_vals / sa_p_vals.sum(dim=-1, keepdim=True).clamp_min(1e-8)
sa_entropy = -torch.sum(sa_p_vals * torch.log(sa_p_vals.clamp_min(1e-8)), dim=-1)
sa_mean_entropy = torch.mean(sa_entropy, dim=(1, 2))

plt.figure(figsize=(5, 5))
plt.plot(torch.exp(sa_mean_entropy).cpu().numpy(), label="Self-Attention Heads", linewidth=2)
plt.xlabel("Layer", fontweight="bold")
plt.xticks(range(num_layers))
plt.ylabel(r"Entropic Rank $\exp\left(-\sum_{i=1}^{n} p_i \log(p_i)\right)$", fontweight="bold")
plt.title("Entropic Rank of Toto Head Outputs", fontweight="bold")
plt.legend()
plt.show()


In [ ]:
def plot_head_similarity_with_magnitude(head_outputs: dict[int, list[torch.Tensor]], output_type: str, save_path: str | None = None):
    n_layers = len(head_outputs)
    if n_layers == 0:
        raise ValueError("No head outputs available to visualize")
    n_heads = len(next(iter(head_outputs.values())))
    ncols = int(np.ceil(np.sqrt(n_layers)))
    nrows = int(np.ceil(n_layers / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axs = np.atleast_1d(axs).flatten()

    for layer_idx in range(n_layers):
        ax = axs[layer_idx]
        heads_data = head_outputs[layer_idx]
        similarities = np.zeros((n_heads, n_heads))
        magnitudes = np.zeros(n_heads)

        for i in range(n_heads):
            head_output = heads_data[i]  # shape: (num_samples, num_timesteps, d_model)
            magnitudes[i] = torch.norm(head_output, dim=-1).mean().item()

        for i in range(n_heads):
            for j in range(n_heads):
                i_flat = heads_data[i].reshape(-1, heads_data[i].shape[-1])
                j_flat = heads_data[j].reshape(-1, heads_data[j].shape[-1])
                similarities[i, j] = torch.nn.functional.cosine_similarity(i_flat, j_flat, dim=-1).mean().item()

        im = ax.imshow(similarities, cmap="RdBu", vmin=-1, vmax=1)
        ax.set_title(f"Layer {layer_idx}")
        ax.set_xlabel("Head Index")
        ax.set_ylabel("Head Index")

        divider = make_axes_locatable(ax)
        ax_mag = divider.append_axes("right", size="15%", pad=0.1)
        ax_mag.barh(np.arange(n_heads), magnitudes, color="coral")
        ax_mag.set_ylim(ax_mag.get_ylim()[::-1])
        ax_mag.set_title("Magnitude")
        ax_mag.set_yticks([])

    for extra_ax in axs[n_layers:]:
        extra_ax.axis("off")

    plt.tight_layout()
    fig.colorbar(im, ax=axs[:n_layers], label="Cosine Similarity", shrink=0.6)
    fig.suptitle(f"{output_type.upper()} Head Output Similarity and Magnitude", fontsize=16)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


In [ ]:
plot_head_similarity_with_magnitude(head_outputs, "sa", f"{plot_save_dir}/sa_head_similarity_toto.png")
plt.show()


In [ ]:
pipeline.remove_all_hooks()
